# Swisstopo Enrichment – Step by Step

Dieses Notebook zerlegt dein `swisstopo-enrich` Script in **einzeln testbare Schritte**. Jeder API-Call (Geocoding, Höhe, Identify, Bevölkerung) ist eine eigene Zelle, sodass du die Antworten der GeoAdmin-Services inspizieren kannst, bevor du über die ganze `listing_details`-Tabelle iterierst.

**Pipeline:**
1. Setup & Konfiguration
2. Helper-Funktionen (Zahlen-Parsing, HTTP-Retry)
3. Geocoding über `SearchServer` → LV95 (E/N)
4. Höhe über `height`-Endpoint
5. Identify – ÖV-Erreichbarkeit & Solar-Einstrahlung
6. Identify – Bevölkerung (Hektarraster)
7. DB-Verbindung (Neon / Postgres) prüfen
8. Eine einzelne Adresse Ende-zu-Ende anreichern
9. Full Run über alle Listings mit Upsert in `swisstopo_details`
10. Ergebnisse verifizieren

> **Credential Hygiene:** Der Script hat den Neon-Connection-String als Fallback hardcoded. Im Notebook lesen wir ihn aus `DATABASE_URL`. Vor einem Commit: Credentials rotieren und nur über `.env` / Secret Store beziehen.


## 1. Imports & Globale Konstanten

Keine Pipeline-Logik, nur Abhängigkeiten. Falls `psycopg2` fehlt:

```bash
pip install psycopg2-binary requests
```


In [ ]:
import os
import math
import time
import json

import requests
import psycopg2
import psycopg2.extras

DEBUG = True  # im Notebook gerne verbose; im Prod-Script stand False


## 2. Konfiguration

- `DATABASE_URL` kommt aus Env (nie im Notebook hardcoden).
- GeoAdmin-Endpoints sind öffentlich, kein API-Key nötig.
- `IDENTIFY_LAYERS` bündelt ÖV + Solar in *einem* Identify-Call, das spart Requests.


In [ ]:
DB_URL = os.environ.get("DATABASE_URL")
assert DB_URL, "Setze DATABASE_URL in der Env, z.B. export DATABASE_URL='postgresql://...'"

SEARCH_URL   = "https://api3.geo.admin.ch/rest/services/api/SearchServer"
IDENTIFY_URL = "https://api3.geo.admin.ch/rest/services/api/MapServer/identify"
HEIGHT_URL   = "https://api3.geo.admin.ch/rest/services/height"

HEADERS = {"User-Agent": "swisstopo-enrich-fast/1.0"}

IDENTIFY_LAYERS = "all:" + ",".join([
    "ch.are.erreichbarkeit-oev",
    "ch.bfe.solarenergie-eignung-daecher",
])

print("DB host:", DB_URL.split("@", 1)[-1].split("/", 1)[0])
print("Layers :", IDENTIFY_LAYERS)


## 3. Helper – Robustes Zahlen-Parsing

GeoAdmin liefert manchmal `None`, leere Strings oder `NaN` – `safe_num` / `safe_int` normalisieren das, damit wir keine `Decimal`-Fehler in Postgres kriegen.


In [ ]:
def safe_num(x, default=None):
    try:
        if x is None:
            return default
        v = float(x)
        if math.isnan(v) or math.isinf(v):
            return default
        return v
    except Exception:
        return default


def safe_int(x, default=None):
    v = safe_num(x, None)
    if v is None:
        return default
    try:
        return int(round(v))
    except Exception:
        return default


# Sanity-Check
assert safe_num("3.14") == 3.14
assert safe_num(None) is None
assert safe_num("nan") is None
assert safe_int("42.7") == 43
print("helpers ok")


## 4. HTTP-Wrapper mit Retry

Ein einziger Retry bei `ConnectionError`. Nicht-200-Responses werden nicht nochmal probiert – das sind meist semantische Fehler (schlechte Geometrie, unbekannter Layer) und ein Retry würde sie nicht heilen.


In [ ]:
def _request(url, *, params=None, timeout=15):
    for attempt in range(2):
        try:
            r = requests.get(url, params=params, timeout=timeout, headers=HEADERS)
            if r.status_code == 200:
                return r
            if attempt == 0 and DEBUG:
                tail = url.rstrip("/").split("/")[-1] or url
                print(f"    ⚠ HTTP {r.status_code} {tail}: {r.text[:160]}")
            return None
        except requests.exceptions.ConnectionError as e:
            if attempt == 0:
                if DEBUG:
                    print(f"    ⚠ connection error, retry in 2s: {e}")
                time.sleep(2)
        except Exception as e:
            if DEBUG:
                print(f"    ⚠ request error: {e}")
            return None
    return None


def _get_json(url, params, timeout=15):
    r = _request(url, params=params, timeout=timeout)
    if not r:
        return None
    try:
        return r.json()
    except Exception:
        return None


## 5. Geocoding (Adresse → LV95 E/N)

Achtung Falle bei GeoAdmin `SearchServer`:
- Feld `y` ist **East** (LV95)
- Feld `x` ist **North** (LV95)

Also nicht der mathematisch-intuitiven X/Y-Benennung folgen, sonst landet jedes Listing im Aargauer Sumpf.


In [ ]:
def geocode(address):
    d = _get_json(SEARCH_URL, {
        "searchText": address,
        "type": "locations",
        "sr": 2056,
        "limit": 1,
    }, timeout=15)

    if not d or not d.get("results"):
        return None

    attrs = d["results"][0].get("attrs", {})
    east  = safe_num(attrs.get("y"))   # y = east
    north = safe_num(attrs.get("x"))   # x = north

    if east is None or north is None:
        return None

    return east, north


# Test: Bundeshaus Bern
test_addr = "Bundesplatz 3, 3003 Bern"
coords = geocode(test_addr)
print(test_addr, "->", coords)
# erwartet: ca. (2600000, 1199500) LV95


## 6. Höhe über `height`-Endpoint

Simpelster Call: E/N rein, Meter über Meer raus. Nützlich für spätere ML-Features (Sicht, Seelage, Skigebiete-Proxy).


In [ ]:
def get_elevation(east, north):
    d = _get_json(HEIGHT_URL, {"easting": east, "northing": north}, timeout=15)
    if not d:
        return None
    return safe_num(d.get("height"))


if coords:
    east, north = coords
    elev = get_elevation(east, north)
    print(f"Höhe {test_addr}: {elev} m ü.M.")  # Bern ~542 m


## 7. Identify – ÖV-Erreichbarkeit & Solar (ein Call)

`identify` ist der Universal-Endpoint: gib ihm Punkt + Layer-Liste, er gibt dir die Attribut-Sätze aller Features zurück, die den Punkt (innerhalb `tolerance`) berühren.

- **ÖV-Score** (`ch.are.erreichbarkeit-oev`, Attr `oev_erreichb_ewap`): höher = besser angebunden
- **Solar** (`ch.bfe.solarenergie-eignung-daecher`, Attr `gstrahlung`): Globalstrahlung kWh/m²

Der Parser nimmt pro Layer das **Maximum** über alle Treffer – z.B. wenn ein Gebäude mehrere Dachflächen hat, interessiert die beste.


In [ ]:
def identify(east, north):
    return _get_json(
        IDENTIFY_URL,
        {
            "geometry": f"{east},{north}",
            "geometryType": "esriGeometryPoint",
            "layers": IDENTIFY_LAYERS,
            "tolerance": 1000,
            "returnGeometry": "false",
            "sr": 2056,
            "imageDisplay": "1000,1000,96",
            "mapExtent": f"{east-1000},{north-1000},{east+1000},{north+1000}",
        },
        timeout=20,
    ) or {"results": []}


def parse_identify(results):
    out = {"oev_score": None, "solar_irr": None}
    for item in results:
        layer = item.get("layerBodId", "")
        attr  = item.get("attributes", {}) or {}

        if layer == "ch.are.erreichbarkeit-oev":
            val = safe_num(attr.get("oev_erreichb_ewap"))
            if val is not None and (out["oev_score"] is None or val > out["oev_score"]):
                out["oev_score"] = val

        elif layer == "ch.bfe.solarenergie-eignung-daecher":
            irr = safe_num(attr.get("gstrahlung"))
            if irr is not None and (out["solar_irr"] is None or irr > out["solar_irr"]):
                out["solar_irr"] = irr
    return out


if coords:
    raw = identify(east, north)
    print(f"# Treffer: {len(raw.get('results') or [])}")
    parsed = parse_identify(raw.get("results") or [])
    print("parsed:", parsed)


## 8. Identify – Bevölkerung (Hektarraster)

`ch.bfs.volkszaehlung-bevoelkerungsstatistik_einwohner` ist ein 100-m-Hektarraster vom BFS. Wir fragen mit sehr kleiner Toleranz und nehmen **nur die erste Zelle** (= die dem Punkt nächste) für das aktuellste Jahr (2024 oder `year=None`-Fallback).

**Wichtig:** Nicht alle Treffer summieren! Wenn der Punkt nahe einer Zellgrenze liegt, liefert GeoAdmin zwei Nachbarzellen zurück (z.B. 44 + 49) – addiert ergäbe das eine Geister-Bevölkerung von 93 statt der tatsächlichen ~44.


In [ ]:
def identify_population(east, north):
    return _get_json(
        IDENTIFY_URL,
        {
            "geometry": f"{east},{north}",
            "geometryType": "esriGeometryPoint",
            "layers": "all:ch.bfs.volkszaehlung-bevoelkerungsstatistik_einwohner",
            "tolerance": 1,                       # kleiner = weniger Grenzfall-Duplikate
            "returnGeometry": "false",
            "sr": 2056,
            "imageDisplay": "100,100,96",
            "mapExtent": f"{east-10},{north-10},{east+10},{north+10}",
        },
        timeout=15,
    ) or {"results": []}


def parse_population(results):
    # Nimmt NUR die nächste Zelle (erster Treffer), summiert NICHT.
    # identify liefert Features nach Nähe sortiert.
    for item in results:
        attr = item.get("attributes", {}) or {}
        number_val = safe_int(attr.get("number"), None)
        year_val   = attr.get("i_year")
        if number_val is None:
            continue
        if year_val is None or year_val == 2024:
            return number_val
    return None


if coords:
    pop_raw = identify_population(east, north)
    hits = pop_raw.get("results") or []
    print(f"# zurückgelieferte Zellen: {len(hits)}")
    for h in hits:
        a = h.get("attributes", {}) or {}
        print(f"   cell={a.get('reli')} year={a.get('i_year')} pop={a.get('number')}")
    pop = parse_population(hits)
    print(f"-> gewählter Wert: {pop}")


## 9. DB-Verbindung prüfen

Kurzer Smoke-Test gegen Neon: Wie viele Listings haben wir, wie viele sind schon angereichert?


In [ ]:
with psycopg2.connect(DB_URL) as _conn:
    with _conn.cursor() as _cur:
        _cur.execute("SELECT COUNT(*) FROM listing_details WHERE address IS NOT NULL;")
        total_listings = _cur.fetchone()[0]
        _cur.execute("SELECT COUNT(*) FROM swisstopo_details;")
        already_enriched = _cur.fetchone()[0]

print(f"Listings mit Adresse : {total_listings}")
print(f"Bereits angereichert : {already_enriched}")
print(f"Offen                : {total_listings - already_enriched}")


## 10. SQL-Statements

- `SELECT_LISTINGS` liest Adressen aus `listing_details`.
- `UPSERT_SQL` schreibt/aktualisiert `swisstopo_details` idempotent – du kannst den Loop also gefahrlos mehrfach laufen lassen.


In [ ]:
SELECT_LISTINGS = """
SELECT listing_id, address
FROM listing_details
WHERE address IS NOT NULL
ORDER BY listing_id;
"""

UPSERT_SQL = """
INSERT INTO swisstopo_details (
    listing_id,
    oev_score, pt_distance_m, solar_irr, elevation_m, population,
    dist_autobahn_m, dist_school_m, lv95_east, lv95_north
) VALUES (
    %s, %s, %s, %s, %s, %s, %s, %s, %s, %s
)
ON CONFLICT (listing_id) DO UPDATE SET
    oev_score       = EXCLUDED.oev_score,
    pt_distance_m   = EXCLUDED.pt_distance_m,
    solar_irr       = EXCLUDED.solar_irr,
    elevation_m     = EXCLUDED.elevation_m,
    population      = EXCLUDED.population,
    dist_autobahn_m = EXCLUDED.dist_autobahn_m,
    dist_school_m   = EXCLUDED.dist_school_m,
    lv95_east       = EXCLUDED.lv95_east,
    lv95_north      = EXCLUDED.lv95_north;
"""
print("SQL ready")


## 11. Eine einzelne Adresse Ende-zu-Ende anreichern

Das ist genau das, was der Loop pro Row macht – hier als Funktion gekapselt, damit du sie *ohne* DB-Write für Debugging aufrufen kannst.


In [ ]:
def enrich_one(address):
    coords = geocode(address)
    if not coords:
        return None
    east, north = coords

    ident  = identify(east, north)
    parsed = parse_identify(ident.get("results") or [])

    pop_ident  = identify_population(east, north)
    population = parse_population(pop_ident.get("results") or [])

    elevation_m = get_elevation(east, north)

    return {
        "lv95_east":       east,
        "lv95_north":      north,
        "oev_score":       parsed["oev_score"],
        "solar_irr":       parsed["solar_irr"],
        "elevation_m":     elevation_m,
        "population":      population,
        "pt_distance_m":   None,   # noch nicht implementiert
        "dist_autobahn_m": None,
        "dist_school_m":   None,
    }


# Probeadresse aus der echten Tabelle
with psycopg2.connect(DB_URL) as _conn, _conn.cursor() as _cur:
    _cur.execute("SELECT listing_id, address FROM listing_details "
                 "WHERE address IS NOT NULL ORDER BY listing_id LIMIT 1;")
    sample_id, sample_addr = _cur.fetchone()

print("Sample:", sample_id, sample_addr)
print(json.dumps(enrich_one(sample_addr), indent=2, ensure_ascii=False))


## 12. Full Run – alle Listings anreichern

Strategie des Original-Scripts:
- **Commit pro Row** (nicht am Ende). Bei Crash verlierst du nur die aktuelle Zeile, nicht alles. Kostet etwas Throughput, aber bei ~hunderten Zeilen irrelevant.
- **`time.sleep(0.05)`** als leichter Rate-Limit-Puffer gegenüber GeoAdmin.
- **Fehler pro Row gefangen**: ein einzelner API-Hänger stoppt nicht den ganzen Run.

Willst du erst mal auf einem Subset testen, setz `LIMIT` in der SELECT-Query.


In [ ]:
def run_enrichment(limit=None, sleep_s=0.05):
    select_sql = SELECT_LISTINGS.rstrip().rstrip(";")
    if limit:
        select_sql += f" LIMIT {int(limit)}"
    select_sql += ";"

    conn = psycopg2.connect(DB_URL)
    conn.autocommit = False
    cur  = conn.cursor()
    dcur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    dcur.execute(select_sql)
    rows = dcur.fetchall()
    total = len(rows)
    print(f"Enriching {total} listings...")

    ok = skip = fail = 0
    try:
        for i, row in enumerate(rows, start=1):
            listing_id = row["listing_id"]
            address    = row["address"]
            try:
                print(f"[{i}/{total}] id={listing_id}")
                coords = geocode(address)
                if not coords:
                    print("   -> geocode failed, skip")
                    skip += 1
                    continue
                east, north = coords

                parsed     = parse_identify(identify(east, north).get("results") or [])
                population = parse_population(identify_population(east, north).get("results") or [])
                elevation_m = get_elevation(east, north)

                cur.execute(UPSERT_SQL, (
                    listing_id,
                    parsed["oev_score"],
                    None,                 # pt_distance_m
                    parsed["solar_irr"],
                    elevation_m,
                    population,
                    None,                 # dist_autobahn_m
                    None,                 # dist_school_m
                    east,
                    north,
                ))
                conn.commit()
                ok += 1
                print(f"   ✓ oev={parsed['oev_score']} solar={parsed['solar_irr']} "
                      f"elev={elevation_m} pop={population}")
                time.sleep(sleep_s)
            except Exception as e:
                conn.rollback()
                fail += 1
                print(f"   ✗ error on id={listing_id}: {e}")
    finally:
        cur.close()
        dcur.close()
        conn.close()

    print(f"Done. ok={ok} skipped={skip} errors={fail}")
    return {"ok": ok, "skipped": skip, "errors": fail}


# Erst mal Dry-Run auf 5 Listings:
# stats = run_enrichment(limit=5)

# Wenn das sauber durchläuft, den echten Run:
# stats = run_enrichment()


## 13. Ergebnisse verifizieren

Schnelle Sanity-Checks nach dem Run:
- Wie viele Zeilen wurden geschrieben?
- Wie viele `NULL`-Werte pro Feld? (Hoher NULL-Anteil bei `solar_irr` ist normal – nicht jede Adresse landet auf einem im BFE-Layer erfassten Dach.)
- Plausibilitäts-Range für Höhe & ÖV.


In [ ]:
with psycopg2.connect(DB_URL) as _conn, _conn.cursor() as _cur:
    _cur.execute("""
        SELECT
            COUNT(*)                                        AS rows_total,
            COUNT(*) FILTER (WHERE oev_score   IS NULL)     AS oev_null,
            COUNT(*) FILTER (WHERE solar_irr   IS NULL)     AS solar_null,
            COUNT(*) FILTER (WHERE elevation_m IS NULL)     AS elev_null,
            COUNT(*) FILTER (WHERE population  IS NULL)     AS pop_null,
            MIN(elevation_m) AS elev_min,
            MAX(elevation_m) AS elev_max,
            MIN(oev_score)   AS oev_min,
            MAX(oev_score)   AS oev_max
        FROM swisstopo_details;
    """)
    row = _cur.fetchone()
    cols = [d[0] for d in _cur.description]

for c, v in zip(cols, row):
    print(f"{c:12s} = {v}")


In [ ]:
# Spot-Check: 5 zufällige angereicherte Datensätze
with psycopg2.connect(DB_URL) as _conn:
    with _conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as _cur:
        _cur.execute("""
            SELECT s.listing_id, l.address,
                   s.lv95_east, s.lv95_north,
                   s.oev_score, s.solar_irr, s.elevation_m, s.population
            FROM swisstopo_details s
            JOIN listing_details   l USING (listing_id)
            ORDER BY random()
            LIMIT 5;
        """)
        for r in _cur.fetchall():
            print(dict(r))


## Nächste Schritte für *Rentables*

Die vier Felder, die im Script als `None` hardcoded sind, sind offene Baustellen:

| Feld              | Mögliche Quelle (GeoAdmin) |
|-------------------|----------------------------|
| `pt_distance_m`   | `ch.bav.haltestellen-oev` – nearest-Point via `identify` + Distance |
| `dist_autobahn_m` | `ch.swisstopo.swissboundaries3d-autobahnen` o.ä. TLM-Layer |
| `dist_school_m`   | `ch.bfs.betriebszaehlung_schulen` oder OSM `amenity=school` |

Für Distanzen: Punkt-zu-Feature-Distance rechnest du am saubersten clientseitig in `shapely` (LV95 ist metrisch, also einfach euklidisch). Alternativ `SELECT ST_Distance(..)` direkt in Postgres, wenn du PostGIS auf der Neon-Instanz aktivierst.

Für das ML-Modell später: `oev_score`, `elevation_m`, `population` + die drei Distanzen sind starke Baseline-Features neben Zimmerzahl / Fläche.
